In [1]:
import numpy as np
import torch
from torch import nn

from utils import HamiltonianOperator

### 2D - spinful
**Note: This is not a formal proof (see OverLeaf for that), just notes for myself to keep in mind when coding.**

Let $\widetilde{\Lambda} = \mathbb{Z}^2 \times \{\uparrow, \downarrow\}$
Then $\Psi: \widetilde{\Lambda}^n \rightarrow \mathbb{C}$ can be written as:
$$\Psi = F_1 g_1 + F_2 g_2,$$
where $F_1 = \Re \{F\}$, $F_2 = \Im \{F\}$, and $F$ is defined by:
$$F(\mathbf{X}) = \prod_{i<j}\left(\widetilde{f}(X_i) - \widetilde{f}(X_j)\right)$$
where $\widetilde{f}: \widetilde{\Lambda} \rightarrow \mathbb{C}$ is injective in the spatial coordinates of the $L\times L$ cell.
Meanwhile, the symmetric functions $g_1, g_2$ are defined by:
$$g_i(\mathbf{X}) = g_i'\left(\sum_{\boldsymbol{s}\in\Lambda} w_{\boldsymbol{s}} \#\{i | X_i = \boldsymbol{s} \, (\bmod L)\, \}\right)$$
where $\Lambda$ is the restriction of the spatial coordinates in $\widetilde{\Lambda}$ to the $L\times L$ cell, i.e. $\Lambda = Z_{L} \times \{\uparrow, \downarrow\}$. Furthermore, because $\Psi$ must be periodic, we require that $\widetilde{f}$ is also periodic. Practically, we can do so by transforming the spatial coordinates into a complex exponential with the appropriate frequency before inputting them into the NN.

In [2]:
# generic MLP helper class
class MLP(nn.Module):
    def __init__(
        self,
        hidden_layers: int,
        input_dim: int,
        dim_feedforward: int,
        output_dim: int,
        activation: nn.Module = nn.GELU,
        device=torch.device("cpu"),
    ):
        super().__init__()
        # make it
        net = [
            nn.Linear(input_dim, dim_feedforward),
            activation(),
        ]
        for _ in range(hidden_layers):
            net.append(nn.Linear(dim_feedforward, dim_feedforward))
            net.append(activation())
        net.append(nn.Linear(dim_feedforward, output_dim))
        self.MLP = nn.Sequential(*net)
        self.to(device)

    # def _initialize_weights(m: nn.Module):
    #     if isinstance(m, nn.Linear):

    def forward(self, x: torch.Tensor):
        return self.MLP(x)


class psi_2D_spinful(nn.Module):
    def __init__(
        self,
        L: int,
        hidden_layers: int,
        dim_feedforward: int,
        activation: nn.Module = nn.GELU,
        device=torch.device("cpu"),
    ):
        super().__init__()
        self.L = L
        # weights, drawn from a Gaussian distribution (for now)
        self.w_s = nn.Parameter(torch.randn(2 * (L**2), device=device))
        # instantiate MLPs
        self.f = MLP(hidden_layers, 5, dim_feedforward, 2, activation, device)
        self.g_1_prime = MLP(hidden_layers, 1, dim_feedforward, 2, activation, device)
        self.g_2_prime = MLP(hidden_layers, 1, dim_feedforward, 2, activation, device)

        self.to(device)

    # antisymmetric factor $F: \widetilde{\Lambda}^n \rightarrow \mathbb{C}$
    def F_antisymmetric(self, x: torch.Tensor):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # convert spatial coords to complex numbers e^{2\pi i x/L}
        z_x = torch.stack(
            [
                torch.cos((2 * torch.pi * x[:, :, 0]) / self.L),
                torch.sin((2 * torch.pi * x[:, :, 0]) / self.L),
                torch.cos((2 * torch.pi * x[:, :, 1]) / self.L),
                torch.sin((2 * torch.pi * x[:, :, 1]) / self.L),
                (2 * x[:, :, 2]) - 1,  # encode spin as ±1
            ],
            dim=-1,
        )  # should be (batch_size, n, 5) now
        # pass through f MLP
        f_x = torch.view_as_complex(self.f(z_x))  # complex (batch_size, n)
        vandermonde_matrix = torch.linalg.vander(
            f_x
        )  # should be of shape (batch_size, n, n)
        # take the determinant
        sign, logabsdet = torch.linalg.slogdet(vandermonde_matrix)
        F = sign * torch.exp(logabsdet)
        return F  # should be a vector of shape (batch_size)

    def eta_symmetric(self, x: torch.Tensor):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # first flatten the last dimension with an injective map
        x = (
            ((2 * self.L) * (x[:, :, 0] % self.L))
            + (2 * (x[:, :, 1] % self.L))
            + x[:, :, 2]
        )
        N_s = (
            nn.functional.one_hot(x, num_classes=2 * (self.L**2)).sum(dim=1).float()
        )  # (batch_size, 2L^2)
        # take matrix-vector product with w_s
        eta = torch.matmul(N_s, self.w_s)
        return eta  # should be a vector of shape (batch_size)

    def forward(self, x: torch.Tensor):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # compute the antisymmetric function F
        F = self.F_antisymmetric(x)  # shape (batch_size), dtype=complex
        # compute the symmetric function g
        # first calculate eta
        eta = self.eta_symmetric(x)  # shape (batch_size), dtype=float
        # now combine them together via:
        # \Psi = F_1 g_1 + F_2 g_2
        g_1 = torch.view_as_complex(
            self.g_1_prime(eta.unsqueeze(-1))
        )  # shape (batch_size), dtype=complex
        g_2 = torch.view_as_complex(self.g_2_prime(eta.unsqueeze(-1)))
        # the final result
        psi = (F.real * g_1) + (F.imag * g_2)  # shape (batch_size), dtype=complex
        return psi

### Test it out
- want antisymmetric

In [3]:
toy_model = psi_2D_spinful(L=2, hidden_layers=3, dim_feedforward=256)
toy_model.eval()

psi_2D_spinful(
  (f): MLP(
    (MLP): Sequential(
      (0): Linear(in_features=5, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): GELU(approximate='none')
      (8): Linear(in_features=256, out_features=2, bias=True)
    )
  )
  (g_1_prime): MLP(
    (MLP): Sequential(
      (0): Linear(in_features=1, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): GELU(approximate='none')
      (8): Linear(in_features=256, out

In [4]:
x = torch.randint(0, 2, (128, 4, 3))
x[:, :, 2] = x[:, :, 2] % 2
x_swapped = x.clone()

In [5]:
# swap two particles
x_swapped[:, [0, 1], :] = x_swapped[:, [1, 0], :]

with torch.no_grad():
    y = toy_model(x_swapped)
print(y[0])

tensor(0.+0.j)


### Energy Minimization Using Gradient Descent for Small Sizes
**Setup:**
Consider a $L \times L$ square lattice in momentum space. The Hubbard Hamiltonian reads:

$\newcommand{\bsl}[1]{\boldsymbol{#1}}$
\begin{align}
    \mathcal{H}_0 &= \sum_{\bsl{k}, \sigma} \varepsilon(\bsl{k}) c_{\bsl{k}\sigma}^{\dagger}c_{\bsl{k}\sigma} \\
    \mathcal{H}_{\text{int}} &= \frac{U}{L^2} \sum_{\bsl{k}, \bsl{k'}, \bsl{q}} c_{\bsl{k}+\bsl{q}, \uparrow}^{\dagger} c_{\bsl{k}, \uparrow} c_{\bsl{k'}-\bsl{q}, \downarrow}^{\dagger} c_{\bsl{k'}, \downarrow} \\
    \mathcal{H} &= \mathcal{H}_0 + \mathcal{H}_{\text{int}},
\end{align}

where $\varepsilon(\bsl{k}) = -2t\left(\cos (k_x) + \cos(k_y)\right)$, $U$ is the on-site repulsion, and $t$ is the hopping parameter. If we convert this Hamiltonian to real space (which is what our NN expects for input), we get:

$$\mathcal{H}=-t\sum_{\langle{\bsl{i}}, \bsl{j}\rangle{},\sigma} \left(c_{\bsl{i}\sigma}^{\dagger} c_{\bsl{j}\sigma} + c_{\bsl{j}\sigma}^{\dagger} c_{\bsl{i}\sigma}\right) + U\sum_{\bsl{i}}n_{\bsl{i}\uparrow}n_{\bsl{i}\downarrow}$$

**Goal:**
We need to minimize
$$E = \frac{\langle{\Psi}|\mathcal{H}|\Psi\rangle{}}{\langle{\Psi}|\Psi\rangle{}}$$

In [6]:
# Check if CUDA is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

Using device: cpu



In [7]:
# test if it's correct via ED
toy_hamiltonian = HamiltonianOperator(2, 2, 0, t=1, U=2)
eigvals, eigvecs = np.linalg.eigh(toy_hamiltonian.H.numpy())
print(f"{eigvals[0]:.12f}")  # matches with my C++ ED!

-7.570521831512


In [8]:
# model parameters
L = 2
n = 2
diff = 0
t = 1
U = 2

hamiltonian = HamiltonianOperator(L, n, diff, t, U)
basis = hamiltonian.basis.to(device)
H_matrix = hamiltonian.H.to(device)

# instantiate the NN model & optimizer
model = psi_2D_spinful(L, hidden_layers=3, dim_feedforward=512, device=device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# now implement the training loop
ED_energy = -7.570521831512  # you can put the value here
NN_energy = float("inf")  # the NN energy
error_criterion = 1e-7
epoch = 0
training_losses = []

while NN_energy - ED_energy > error_criterion:
    # set to training mode
    model.train()

    # compute Psi
    Psi = model(basis)

    # the expectation value
    E = torch.vdot(Psi, H_matrix @ Psi) / torch.vdot(Psi, Psi)

    # backpropagation
    E.real.backward()
    optimizer.step()
    optimizer.zero_grad()

    # logging
    NN_energy = E.real.item()
    training_losses.append(NN_energy)
    print(f"Epoch {epoch}, E={NN_energy:.6f}")
    epoch += 1

Epoch 0, E=-7.008049
Epoch 1, E=-6.753509
Epoch 2, E=-7.294147
Epoch 3, E=-7.378959
Epoch 4, E=-7.353668
Epoch 5, E=-7.407207
Epoch 6, E=-7.450686
Epoch 7, E=-7.458970
Epoch 8, E=-7.455245
Epoch 9, E=-7.457244
Epoch 10, E=-7.467561
Epoch 11, E=-7.480828
Epoch 12, E=-7.490967
Epoch 13, E=-7.495383
Epoch 14, E=-7.495683
Epoch 15, E=-7.495359
Epoch 16, E=-7.496745
Epoch 17, E=-7.499781
Epoch 18, E=-7.502927
Epoch 19, E=-7.504756
Epoch 20, E=-7.504935
Epoch 21, E=-7.504192
Epoch 22, E=-7.503645
Epoch 23, E=-7.504058
Epoch 24, E=-7.505472
Epoch 25, E=-7.507315
Epoch 26, E=-7.508811
Epoch 27, E=-7.509466
Epoch 28, E=-7.509301
Epoch 29, E=-7.508771
Epoch 30, E=-7.508429
Epoch 31, E=-7.508574
Epoch 32, E=-7.509136
Epoch 33, E=-7.509806
Epoch 34, E=-7.510300
Epoch 35, E=-7.510536
Epoch 36, E=-7.510642
Epoch 37, E=-7.510819
Epoch 38, E=-7.511184
Epoch 39, E=-7.511699
Epoch 40, E=-7.512198
Epoch 41, E=-7.512513
Epoch 42, E=-7.512576
Epoch 43, E=-7.512452
Epoch 44, E=-7.512290
Epoch 45, E=-7.51222